# Résumé de ce qu'on a fait

In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns
import plotly.express as px
from plotly.offline import plot
import plotly.io as pio
from sklearn.manifold import MDS
import librosa
from scipy.cluster.hierarchy import linkage, dendrogram
import plotly.graph_objects as go
import os
import warnings
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import accuracy_score
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [35]:
li_filenames = []

for racine, _, fichiers in os.walk('cross-era_chroma-nnls'):
    for fichier in fichiers:
        chemin_relatif = os.path.relpath(os.path.join(racine, fichier))
        li_filenames.append(chemin_relatif)

In [36]:
def periode(str):
    try:
        date_deb = int(str[0:4])
        date_fin = int(str[5:9])
        date = (date_deb + date_fin)/2
        if date < 1500:
            return "Moyen-Âge"
        elif date < 1600:
            return "Renaissance"
        elif date < 1750:
            return "Baroque"
        elif date < 1800:
            return "Classique"
        elif date < 1880:
            return "Romantique"
        else:
            return "XXe siècle"
    except :
        return "Inconnu"

In [37]:
def dico_filenames(filename = "cross-era_annotations.csv"):
    dico = {}
    dico2 = {}
    df = pd.read_csv(filename, sep=',')
    for i in range(len(df)):
        dico[df['Filename'][i]] = df['Composer'][i]
        dico2[df['Filename'][i]] = periode(df['CompLifetime'][i])
    return dico, dico2

In [38]:
pd.read_csv('cross-era_annotations.csv', sep=',')

,Class,Filename,CrossEra-ID,Instrumentation,Key,Mode,Composer,CompLifetime,Country,Unnamed: 9
0,orchestra_baroque,CrossEra-0001_Albinoni__sinata_a_cinque_no._6_...,CrossEra-0001,orchestra,G,minor,Albinoni; Tomaso,1671-1751,Italy,NaN
1,orchestra_baroque,CrossEra-0002_Albinoni__sinata_a_cinque_no._6_...,CrossEra-0002,orchestra,G,minor,Albinoni; Tomaso,1671-1751,Italy,NaN
2,orchestra_baroque,CrossEra-0003_Albinoni__sinata_a_cinque_no._6_...,CrossEra-0003,orchestra,G,minor,Albinoni; Tomaso,1671-1751,Italy,NaN
3,orchestra_baroque,CrossEra-0004_Albinoni__sinata_a_cinque_no._6_...,CrossEra-0004,orchestra,G,minor,Albinoni; Tomaso,1671-1751,Italy,NaN
4,orchestra_baroque,CrossEra-0005_Albinoni_concerto_in_a_minor_bwv...,CrossEra-0005,orchestra,A,minor,Albinoni; Tomaso,1671-1751,Italy,NaN
...,...,...,...,...,...,...,...,...,...,...
1995,piano_addon,CrossEra-1996_Weber_sonata_no._35_in_a_minor_o...,CrossEra-1996,piano,NaN,NaN,Weber; Carl Maria von,1786-1826,Germany,NaN
1996,piano_addon,CrossEra-1997_Weber_sonata_no._4_in_e_minor_op...,CrossEra-1997,piano,NaN,NaN,Weber; Carl Maria von,1786-1826,Germany,NaN
1997,piano_addon,CrossEra-1998_Weber_sonata_no._4_in_e_minor_op...,CrossEra-1998,piano,NaN,NaN,Weber; Carl Maria von,1786-1826,Germany,NaN
1998,piano_addon,CrossEra-1999_Weber_sonata_no._4_in_e_minor_op...,CrossEra-1999,piano,NaN,NaN,Weber; Carl Maria von,1786-1826,Germany,NaN


In [39]:
dico_composers, dico_periodes = dico_filenames()
a = np.array(list(dico_composers.values()))
np.unique(a)
# Imprime toutes les clés ayant pour valeur 'major'
for key, value in dico_composers.items():
    if value == ' major':
        print(key)
dico_composers["CrossEra-0616_Borodin_symphony_no.3_in_a_minor_moderato_assai.mp3"] = 'Borodin; Alexander'
dico_composers["CrossEra-0673_Liszt_poems__mazeppa.mp3"] = 'Liszt; Franz'
dico_composers["CrossEra-0674_Liszt_poems__prometheus.mp3"] = 'Liszt; Franz'
dico_composers["CrossEra-0776_Verdi_Overt_giovanna_darco_sinfonia.mp3"] = 'Verdi; Giuseppe'
dico_composers["CrossEra-1016_Cimarosa_Piano_sonata_no._24_in_b-flat_minor_major___andantino.mp3"] = 'Cimarosa; Domenico'

CrossEra-0616_Borodin_symphony_no.3_in_a_minor_moderato_assai.mp3
CrossEra-0673_Liszt_poems__mazeppa.mp3
CrossEra-0674_Liszt_poems__prometheus.mp3
CrossEra-0776_Verdi_Overt_giovanna_darco_sinfonia.mp3
CrossEra-1016_Cimarosa_Piano_sonata_no._24_in_b-flat_minor_major___andantino.mp3


In [40]:
# Load the data
with open("hist_cross-era.pkl", "rb") as f:
    dico_cross_era = pickle.load(f)

In [41]:
X = []
etiquettes = []
y = []
periode = []
for file, histograms in dico_cross_era.items():
    composer = dico_composers[file.split('/')[1]]
    etiquettes.append(file)
    matrice = np.vstack(histograms)
    X.append(matrice.flatten())
    y.append(composer)
    periode.append(dico_periodes[file.split('/')[1]])
X = np.array(X)
X_scaled = StandardScaler().fit_transform(X)
y = np.array(y)
periodes = np.array(periode)
morceaux = np.array(etiquettes)

# Première analyse des données

In [42]:
import umap.umap_ as umap
from sklearn.preprocessing import LabelEncoder

reducer_3d = umap.UMAP(n_neighbors=4, min_dist=0.0, n_components=3, metric='cosine')
le = LabelEncoder()
y_numeric = le.fit_transform(y)
X_umap_3d = reducer_3d.fit_transform(X_scaled, y=y_numeric)

fig = px.scatter_3d(
    x=X_umap_3d[:, 0],
    y=X_umap_3d[:, 1],
    z=X_umap_3d[:, 2],
    color=y,
    labels={'color': 'Compositeur'},
    hover_name=morceaux,
    hover_data={'Compositeur': y, 'Période': periodes},
    title="Projection UMAP 3D des morceaux"
)
fig.update_traces(marker=dict(size=4))
fig.show()


In [43]:
# Calculer les centroïdes pour chaque compositeur
# On retire la colonne 'periode' avant de faire la moyenne
df_X = pd.DataFrame(X_umap_3d)
df_X['composer'] = y
centroides = df_X.groupby('composer').mean()
df_X['periode'] = periodes

# Récupérer la période dominante pour chaque compositeur
periode_centroide = df_X.groupby('composer')['periode'].agg(lambda x: x.value_counts().idxmax())

# Affichage des centroïdes sur plotly (projection PCA 3D pour visualisation)
centroides_pca = centroides.values

fig = px.scatter_3d(
    x=centroides_pca[:, 0],
    y=centroides_pca[:, 1],
    z=centroides_pca[:, 2],
    # text=centroides.index,
    hover_name=centroides.index,
    color=periode_centroide,
    title="Centroïdes des compositeurs (PCA 3D, coloré par période)",
    labels={'x': 'PCA1', 'y': 'PCA2', 'z': 'PCA3', 'color': 'Période'}
)
fig.update_traces(marker=dict(size=8))
fig.show()

In [44]:
from sklearn.manifold import TSNE

# Projection T-SNE 3D
tsne_3d = TSNE(n_components=3, perplexity=100, random_state=42, metric='cosine')
X_tsne_3d = tsne_3d.fit_transform(X_scaled)

fig = px.scatter_3d(
    x=X_tsne_3d[:, 0],
    y=X_tsne_3d[:, 1],
    z=X_tsne_3d[:, 2],
    color=y,
    labels={'color': 'Compositeur'},
    hover_name=morceaux,
    title="Projection T-SNE 3D des morceaux"
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [45]:
# Calculer les centroïdes pour chaque compositeur
# On retire la colonne 'periode' avant de faire la moyenne
df_X = pd.DataFrame(X_tsne_3d)
df_X['composer'] = y
centroides = df_X.groupby('composer').mean()
df_X['periode'] = periodes

# Récupérer la période dominante pour chaque compositeur
periode_centroide = df_X.groupby('composer')['periode'].agg(lambda x: x.value_counts().idxmax())

# Affichage des centroïdes sur plotly (projection PCA 3D pour visualisation)
centroides_pca = centroides.values

fig = px.scatter_3d(
    x=centroides_pca[:, 0],
    y=centroides_pca[:, 1],
    z=centroides_pca[:, 2],
    # text=centroides.index,
    hover_name=centroides.index,
    color=periode_centroide,
    title="Centroïdes des compositeurs (PCA 3D, coloré par période)",
    labels={'x': 'PCA1', 'y': 'PCA2', 'z': 'PCA3', 'color': 'Période'}
)
fig.update_traces(marker=dict(size=8))
fig.show()

In [46]:
from sklearn.decomposition import PCA

# PCA 3D
pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X_scaled)

fig = px.scatter_3d(
    x=X_pca_3d[:, 0],
    y=X_pca_3d[:, 1],
    z=X_pca_3d[:, 2],
    color=y,
    labels={'color': 'Compositeur'},
    hover_name=morceaux,
    hover_data={'Compositeur': y, 'Période': periodes},
    title="Projection PCA 3D des morceaux"
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [47]:
# Calculer les centroïdes pour chaque compositeur
# On retire la colonne 'periode' avant de faire la moyenne
df_X = pd.DataFrame(X_scaled)
df_X['composer'] = y
centroides = df_X.groupby('composer').mean()
df_X['periode'] = periodes

# Récupérer la période dominante pour chaque compositeur
periode_centroide = df_X.groupby('composer')['periode'].agg(lambda x: x.value_counts().idxmax())

# Affichage des centroïdes sur plotly (projection PCA 3D pour visualisation)
centroides_pca = pca_3d.transform(centroides.values)

fig = px.scatter_3d(
    x=centroides_pca[:, 0],
    y=centroides_pca[:, 1],
    z=centroides_pca[:, 2],
    # text=centroides.index,
    hover_name=centroides.index,
    color=periode_centroide,
    title="Centroïdes des compositeurs (PCA 3D, coloré par période)",
    labels={'x': 'PCA1', 'y': 'PCA2', 'z': 'PCA3', 'color': 'Période'}
)
fig.update_traces(marker=dict(size=8))
fig.show()

In [48]:
# PCA réduction à 500 dimensions
pca_500 = PCA(n_components=1000)
X_pca_500 = pca_500.fit_transform(X_scaled)

# UMAP sur la réduction PCA
reducer_pca_umap = umap.UMAP(n_neighbors=10, min_dist=0.0, n_components=3, metric='cosine')
X_pca_umap_3d = reducer_pca_umap.fit_transform(X_pca_500, y=y_numeric)

fig = px.scatter_3d(
    x=X_pca_umap_3d[:, 0],
    y=X_pca_umap_3d[:, 1],
    z=X_pca_umap_3d[:, 2],
    color=y,
    labels={'color': 'Compositeur'},
    hover_name=morceaux,
    title="PCA (500) + UMAP 3D des morceaux"
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [49]:
# Calculer les centroïdes pour chaque compositeur
# On retire la colonne 'periode' avant de faire la moyenne
df_X = pd.DataFrame(X_pca_umap_3d)
df_X['composer'] = y
centroides = df_X.groupby('composer').mean()
df_X['periode'] = periodes

# Récupérer la période dominante pour chaque compositeur
periode_centroide = df_X.groupby('composer')['periode'].agg(lambda x: x.value_counts().idxmax())

# Affichage des centroïdes sur plotly (projection PCA 3D pour visualisation)
centroides_pca = centroides.values

fig = px.scatter_3d(
    x=centroides_pca[:, 0],
    y=centroides_pca[:, 1],
    z=centroides_pca[:, 2],
    # text=centroides.index,
    hover_name=centroides.index,
    color=periode_centroide,
    title="Centroïdes des compositeurs (PCA 3D, coloré par période)",
    labels={'x': 'PCA1', 'y': 'PCA2', 'z': 'PCA3', 'color': 'Période'}
)
fig.update_traces(marker=dict(size=8))
fig.show()

## Clustering

In [ ]:
lda_3d = LDA(n_components=3)
X_lda_3d = lda_3d.fit_transform(X_scaled, y)

fig = px.scatter_3d(
    x=X_lda_3d[:, 0],
    y=X_lda_3d[:, 1],
    z=X_lda_3d[:, 2],
    color=y,
    labels={'color': 'Compositeur'},
    hover_name=morceaux,
    hover_data={'Compositeur': y, 'Période': periodes},
    title="Projection LDA 3D des morceaux"
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [ ]:
# Calculer les centroïdes pour chaque compositeur
# On retire la colonne 'periode' avant de faire la moyenne
df_X = pd.DataFrame(X_scaled)
df_X['composer'] = y
centroides = df_X.groupby('composer').mean()
df_X['periode'] = periodes

# Récupérer la période dominante pour chaque compositeur
periode_centroide = df_X.groupby('composer')['periode'].agg(lambda x: x.value_counts().idxmax())

# Affichage des centroïdes sur plotly (projection PCA 3D pour visualisation)
centroides_pca = lda_3d.transform(centroides.values)

fig = px.scatter_3d(
    x=centroides_pca[:, 0],
    y=centroides_pca[:, 1],
    z=centroides_pca[:, 2],
    # text=centroides.index,
    hover_name=centroides.index,
    color=periode_centroide,
    title="Centroïdes des compositeurs (PCA 3D, coloré par période)",
    labels={'x': 'PCA1', 'y': 'PCA2', 'z': 'PCA3', 'color': 'Période'}
)
fig.update_traces(marker=dict(size=8))
fig.show()

## Test de l'espace de représentation

In [ ]:
# Load the data
with open("dico_hist_supp.pkl", "rb") as f:
    dico_supp = pickle.load(f)

In [ ]:
Xsup = []
etiquettessup = []
ysup = []
periodesup = []
for file, histograms in dico_supp.items():
    etiquettessup.append(file)
    matrice = np.vstack(histograms)
    Xsup.append(matrice.flatten())
    ysup.append(file)
    periodesup.append(file)
X = np.vstack([X, Xsup])
y = np.concatenate([y, ysup])
morceaux = np.concatenate([morceaux, etiquettessup])
periodes = np.concatenate([periodes, periodesup])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
lda_3d = LDA(n_components=3)
X_lda_3d = lda_3d.fit_transform(X_scaled, y)

fig = px.scatter_3d(
    x=X_lda_3d[:, 0],
    y=X_lda_3d[:, 1],
    z=X_lda_3d[:, 2],
    color=y,
    labels={'color': 'Compositeur'},
    hover_name=morceaux,
    hover_data={'Compositeur': y, 'Période': periodes},
    title="Projection LDA 3D des morceaux"
)
fig.update_traces(marker=dict(size=4))
fig.show()